# Ejercicio 4 - Analisis exploratorio con DuckDB

Todas las consultas usan las vistas de `sql/00_vistas.sql` (`viajes`, `viajes_enriquecidos`, `viajes_validos`, `zonas`), que leen los Parquet al momento de consultar. Como no mencionan rutas ni anios, siguen funcionando al agregar 2024 y 2025.

Preguntas, justificacion e interpretacion: `docs/ejercicio4_analisis.md`. SQL + resultados + tiempos: `python scripts/run_sql.py ejercicio4`. Las figuras se guardan en `docs/figuras/`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "scripts"))   # notebooks/ -> scripts/
from lab import conectar, ejecutar_sql, consulta, RAIZ_REPO

import pandas as pd
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

def correr(archivo):
    """Ejecuta un .sql, muestra sus metadatos y tiempo, y devuelve el DataFrame."""
    meta, df, seg = ejecutar_sql(con, archivo)
    print(f"[{meta.get('id')}] {meta.get('titulo', '')}")
    for clave in ("pregunta", "objetivo", "fuente"):
        if clave in meta:
            print(f"  {clave}: {meta[clave]}")
    print(f"  tiempo: {seg:.3f} s | filas: {len(df)}")
    return df

con = conectar()
import matplotlib.pyplot as plt
import numpy as np

DIR_FIG = RAIZ_REPO / "docs" / "figuras"
DIR_FIG.mkdir(parents=True, exist_ok=True)
COLOR = {"yellow": "#E8A202", "green": "#2E8B57"}
DIAS = ["Lun", "Mar", "Mie", "Jue", "Vie", "Sab", "Dom"]
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

def guardar(fig, nombre):
    fig.tight_layout()
    fig.savefig(DIR_FIG / f"{nombre}.png", bbox_inches="tight")
    plt.show()


## P1 - Volumen mensual por tipo de taxi

In [ ]:
p1 = correr("sql/ejercicio4/4_01_viajes_por_mes.sql")
fig, ejes = plt.subplots(1, 2, figsize=(12, 3.8))
for eje, taxi in zip(ejes, ["yellow", "green"]):
    d = p1[p1.taxi == taxi]
    eje.plot(d.mes, d.viajes_por_dia, marker="o", color=COLOR[taxi])
    eje.set_title(f"{taxi}: viajes promedio por dia")
    eje.set_ylim(bottom=0)
    eje.tick_params(axis="x", rotation=45)
guardar(fig, "p1_viajes_por_mes")
p1

## P2 - Demanda por hora y dia de la semana

Color = % de los viajes de ese tipo de taxi (escala propia por tipo).

In [ ]:
p2 = correr("sql/ejercicio4/4_02_hora_dia_semana.sql")
fig, ejes = plt.subplots(1, 2, figsize=(13, 3.8))
for eje, taxi in zip(ejes, ["yellow", "green"]):
    m = p2[p2.taxi == taxi].pivot(index="dia_semana", columns="hora", values="pct_del_tipo")
    im = eje.imshow(m.values, aspect="auto", cmap="YlOrRd" if taxi == "yellow" else "YlGn")
    eje.set_yticks(range(7), DIAS)
    eje.set_xticks(range(0, 24, 2), range(0, 24, 2))
    eje.set_xlabel("hora de pickup")
    eje.set_title(taxi)
    fig.colorbar(im, ax=eje, label="% de viajes")
guardar(fig, "p2_hora_dia")
p2.groupby(["taxi", "hora"]).viajes.sum().unstack(0).idxmax()   # hora pico por tipo

## P3 - Caracteristicas de los viajes

Linea = p10 a p90, caja = p25 a p75, punto = mediana.

In [ ]:
p3 = correr("sql/ejercicio4/4_03_caracteristicas_viaje.sql")
variables = p3.variable.unique()
fig, ejes = plt.subplots(1, len(variables), figsize=(14, 2.8))
for eje, var in zip(ejes, variables):
    d = p3[p3.variable == var].reset_index(drop=True)
    for i, r in d.iterrows():
        c = COLOR[r.taxi]
        eje.hlines(i, r.p10, r.p90, color=c, lw=1.5)
        eje.hlines(i, r.p25, r.p75, color=c, lw=7, alpha=.6)
        eje.plot(r.mediana, i, "o", color="black", ms=5)
    eje.set_yticks(range(len(d)), d.taxi)
    eje.set_ylim(-0.6, len(d) - 0.4)
    eje.set_title(var)
guardar(fig, "p3_caracteristicas")
p3

## P4 - Comparacion general amarillos vs. verdes

In [ ]:
p4 = correr("sql/ejercicio4/4_04_resumen_por_tipo.sql")
p4.set_index("taxi").T

## P5 - Origen por borough

In [ ]:
p5 = correr("sql/ejercicio4/4_05_viajes_por_borough.sql")
m = p5.pivot(index="taxi", columns="borough_origen", values="pct_del_tipo").fillna(0)
m = m[m.sum().sort_values(ascending=False).index]
fig, eje = plt.subplots(figsize=(11, 2.6))
izq = np.zeros(len(m))
for col in m.columns:
    eje.barh(m.index, m[col], left=izq, label=col)
    izq += m[col].values
eje.set_xlabel("% de los viajes del tipo")
eje.legend(ncol=4, bbox_to_anchor=(0.5, -0.35), loc="upper center", frameon=False)
guardar(fig, "p5_borough")
m.round(2)

## P6 - Metodo de pago y propina registrada

In [ ]:
p6 = correr("sql/ejercicio4/4_06_metodo_pago.sql")
fig, ejes = plt.subplots(1, 2, figsize=(12, 3.6))
for eje, col, titulo in [(ejes[0], "pct_del_tipo", "% de viajes por metodo"),
                         (ejes[1], "pct_con_propina", "% de viajes con propina > 0")]:
    m = p6.pivot(index="metodo_pago", columns="taxi", values=col).reindex(columns=["yellow", "green"])
    m.plot.bar(ax=eje, color=[COLOR["yellow"], COLOR["green"]], rot=0)
    eje.set_title(titulo); eje.set_xlabel("")
guardar(fig, "p6_pago")
p6

## P7 - Porcentaje de propina (pagos con tarjeta)

In [ ]:
p7 = correr("sql/ejercicio4/4_07_distribucion_propina.sql")
orden = p7.sort_values("orden").rango_propina.unique()
m = p7.pivot(index="rango_propina", columns="taxi", values="pct_del_tipo").reindex(orden)
m = m.reindex(columns=[c for c in ["yellow", "green"] if c in m.columns])
fig, eje = plt.subplots(figsize=(10, 3.4))
m.plot.bar(ax=eje, color=[COLOR[c] for c in m.columns], rot=0)
eje.set_ylabel("% de viajes con tarjeta"); eje.set_xlabel("propina / tarifa")
guardar(fig, "p7_propina")
m

## P8 - Distribucion del monto total

In [ ]:
p8 = correr("sql/ejercicio4/4_08_histograma_total.sql")
fig, eje = plt.subplots(figsize=(11, 3.4))
for taxi in ["yellow", "green"]:
    d = p8[p8.taxi == taxi]
    eje.step(d.desde_usd, d.pct_del_tipo, where="post", color=COLOR[taxi], label=taxi, lw=2)
eje.set_xlabel("total del viaje (USD, intervalos de 5; 150 = 150 o mas)")
eje.set_ylabel("% de viajes"); eje.legend()
guardar(fig, "p8_histograma_total")

## P9 - Registros atipicos o inconsistentes por regla

In [ ]:
p9 = correr("sql/ejercicio4/4_09_impacto_reglas_calidad.sql")
reglas = ["fuera_periodo", "duracion", "distancia", "velocidad", "monto", "pasajeros"]
fig, ejes = plt.subplots(1, 2, figsize=(13, 3.6))
for taxi in ["yellow", "green"]:
    d = p9[p9.taxi == taxi]
    ejes[0].plot(d.mes_archivo, d.pct_excluido, marker="o", color=COLOR[taxi], label=taxi)
ejes[0].set_title("% de registros excluidos por mes"); ejes[0].legend()
ejes[0].tick_params(axis="x", rotation=45); ejes[0].set_ylim(bottom=0)
tot = p9.groupby("taxi")[reglas + ["registros"]].sum()
pct_regla = tot[reglas].div(tot.registros, axis=0) * 100
pct_regla.T[["yellow", "green"]].plot.bar(ax=ejes[1], color=[COLOR["yellow"], COLOR["green"]], rot=0)
ejes[1].set_title("% de registros que viola cada regla")
guardar(fig, "p9_calidad")
pct_regla.round(3)

## P10 - Atipicos de costo por milla (IQR)

In [ ]:
correr("sql/ejercicio4/4_10_atipicos_tarifa_milla.sql")